# Planwise: CMS Data Integration Review
**Rechelle Transeth — November 7, 2026**

Scope: Michigan stand-alone Medicare Part D plans, Q3 2026. This review documents complete component downloads, cleaned Michigan tables, join checks, medication-name mapping, and the intake data contract. It does not implement or validate a complete annual cost engine.

Open from the `planwise_data` folder, select the Python environment containing DuckDB and Jupyter, then Run All. Saved outputs support review without running code. Rebuilding the database uses the scripts described in the README; this notebook performs read-only validation against the saved database.

In [1]:
from pathlib import Path
import json, html, sys
import duckdb
from IPython.display import display, HTML, Markdown
ROOT=Path.cwd()
if not (ROOT/'data/planwise.duckdb').exists(): ROOT=ROOT/'planwise_data'
assert (ROOT/'data/planwise.duckdb').exists(), 'Open from planwise_data after building the database.'
sys.path.insert(0,str(ROOT/'scripts'))
from lookup import match_medication
E=ROOT/'evidence'
def table(rows):
    if not rows:
        print('No records.');return
    headers=list(rows[0]);body=''.join('<tr>'+''.join('<td>'+html.escape(str(r.get(k,'')))+'</td>' for k in headers)+'</tr>' for r in rows)
    display(HTML('<div style="overflow-x:auto"><table><thead><tr>'+''.join('<th>'+html.escape(k)+'</th>' for k in headers)+'</tr></thead><tbody>'+body+'</tbody></table></div>'))
def query(sql,params=None):
    with duckdb.connect(str(ROOT/'data/planwise.duckdb'),read_only=True) as c:
        cur=c.execute(sql,params or [])
        return [dict(zip([x[0] for x in cur.description],r)) for r in cur.fetchall()]
print('DuckDB',duckdb.__version__)

DuckDB 2.0.0


## 1. Required files and provenance

The full pricing component and all six pharmacy-network components are downloaded, alongside plan, geographic, basic formulary, cost-sharing, insulin, excluded-drug and indication files. Original ZIPs are retained. The table distinguishes nationwide scanned rows from Michigan-scoped rows. Benefit year is 2026; the release covers July–September and was published in October.

[CMS quarterly data source](https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/quarterly-prescription-drug-plan-formulary-pharmacy-network-and-pricing-information) · [CMS field definitions](https://data.cms.gov/sites/default/files/2025-10/83da019f-fa24-483e-87de-cc089780a6a5/SPUFRecordLayout-2026.pdf)

In [2]:
manifest=json.loads((E/'download_manifest.json').read_text())
table([{k:r[k] for k in ['file','bytes','plan_year','release']} for r in manifest])
audit=json.loads((E/'load_audit.json').read_text())
table(audit['sources'])
table([dict(table=k,raw_scoped_rows=v['raw_scoped_rows'],clean_rows=v['clean_rows'],exact_duplicates=v['exact_duplicates']) for k,v in audit['tables'].items()])

file,bytes,plan_year,release
basic drugs formulary file PPUF_2026Q3.zip,8605275,2026,Q3 2026
beneficiary cost file PPUF_2026Q3.zip,649365,2026,Q3 2026
excluded drugs formulary file PPUF_2026Q3.zip,36182,2026,Q3 2026
geographic locator file PPUF_2026Q3.zip,29615,2026,Q3 2026
indication based coverage formulary file PPUF_2026Q3.zip,1745,2026,Q3 2026
insulin beneficiary cost file PPUF_2026Q3.zip,163592,2026,Q3 2026
pharmacy networks file PPUF_2026Q3 part 1.zip,409403545,2026,Q3 2026
pharmacy networks file PPUF_2026Q3 part 2.zip,408375712,2026,Q3 2026
pharmacy networks file PPUF_2026Q3 part 3.zip,409567344,2026,Q3 2026
pharmacy networks file PPUF_2026Q3 part 4.zip,407278852,2026,Q3 2026


table,archive,member,source_rows,extracted_bytes
geography,geographic locator file PPUF_2026Q3.zip,geographic locator file PPUF_2026Q3.txt,3279,303047
plans,plan information PPUF_2026Q3.zip,plan information PPUF_2026Q3.txt,112294,13689499
formulary,basic drugs formulary file PPUF_2026Q3.zip,basic drugs formulary file PPUF_2026Q3.txt,1129364,59052703
cost_sharing,beneficiary cost file PPUF_2026Q3.zip,beneficiary cost file PPUF_2026Q3.txt,172549,10149436
insulin_cost,insulin beneficiary cost file PPUF_2026Q3.zip,insulin beneficiary cost file PPUF_2026Q3.txt,43045,2365343
pricing,pricing file PPUF_2026Q3.zip,pricing file PPUF_2026Q3.txt,54052371,2021334092
pharmacy_network,pharmacy networks file PPUF_2026Q3 part 1.zip,pharmacy networks file PPUF_2026Q3 part 1.txt,54943624,4113536748
pharmacy_network,pharmacy networks file PPUF_2026Q3 part 2.zip,pharmacy networks file PPUF_2026Q3 part 2.txt,54765701,4116151652
pharmacy_network,pharmacy networks file PPUF_2026Q3 part 3.zip,pharmacy networks file PPUF_2026Q3 part 3.txt,54592539,4078772616
pharmacy_network,pharmacy networks file PPUF_2026Q3 part 4.zip,pharmacy networks file PPUF_2026Q3 part 4.txt,54822899,4160764590


table,raw_scoped_rows,clean_rows,exact_duplicates
geography,3279,3279,0
plans,12,12,0
formulary,35441,35441,0
cost_sharing,330,330,0
insulin_cost,88,88,0
pricing,100111,100111,0
pharmacy_network,667631,667631,0
excluded_drugs,18,18,0
indications,0,0,0


## 2. Quality checks and schemas

Identifiers remain strings; column names and whitespace are normalized. Numeric conversions are audited. Exact duplicates are removed, while conflicting logical keys are reported rather than arbitrarily resolved. Cost-sharing days-supply codes are mapped explicitly; code 3 (other) is left unresolved. Missing values are not zero costs.

In [3]:
checks=json.loads((E/'quality_checks.json').read_text())
table([{'check':k,'result':v} for k,v in checks.items()])
for name,stats in audit['tables'].items():
    display(Markdown('**'+name+'**'))
    table([{'field':col,'type':typ,'missing_rows':stats['null_counts'].get(col,0),'numeric_parse_failures':stats['numeric_conversion_failures'].get(col,0),'source_period_markers':stats.get('numeric_missing_markers',{}).get(col,0)} for col,typ in stats['schema']])

check,result
plans_duplicate_keys,0
plans_missing_keys,0
formulary_duplicate_keys,0
formulary_missing_keys,0
pricing_duplicate_keys,0
pricing_missing_keys,0
cost_sharing_duplicate_keys,0
cost_sharing_missing_keys,0
pharmacy_network_duplicate_keys,0
pharmacy_network_missing_keys,0


**geography**

field,type,missing_rows,numeric_parse_failures,source_period_markers
county_code,VARCHAR,0,0,0
statename,VARCHAR,0,0,0
county,VARCHAR,0,0,0
ma_region_code,VARCHAR,118,0,0
ma_region,VARCHAR,118,0,0
pdp_region_code,VARCHAR,0,0,0
pdp_region,VARCHAR,0,0,0


**plans**

field,type,missing_rows,numeric_parse_failures,source_period_markers
contract_id,VARCHAR,0,0,0
plan_id,VARCHAR,0,0,0
segment_id,VARCHAR,0,0,0
contract_name,VARCHAR,0,0,0
plan_name,VARCHAR,0,0,0
formulary_id,VARCHAR,0,0,0
premium,"DECIMAL(18,6)",0,0,0
deductible,"DECIMAL(18,6)",0,0,0
ma_region_code,VARCHAR,12,0,0
pdp_region_code,VARCHAR,0,0,0


**formulary**

... (9 outputs left out)

**excluded_drugs**

field,type,missing_rows,numeric_parse_failures,source_period_markers
contract_id,VARCHAR,0,0,0
plan_id,VARCHAR,0,0,0
rxcui,VARCHAR,0,0,0
tier,INTEGER,0,0,0
quantity_limit_yn,VARCHAR,0,0,0
quantity_limit_amount,VARCHAR,7,0,0
quantity_limit_days,VARCHAR,7,0,0
prior_auth_yn,VARCHAR,0,0,0
step_therapy_yn,VARCHAR,0,0,0
capped_benefit_yn,VARCHAR,0,0,0


**indications**

field,type,missing_rows,numeric_parse_failures,source_period_markers
contract_id,VARCHAR,0,0,0
plan_id,VARCHAR,0,0,0
rxcui,VARCHAR,0,0,0
disease,VARCHAR,0,0,0


## 3. Coverage of Michigan plans

The selection uses S contracts and the Michigan PDP region (13), not the local-MA STATE field. The coverage table checks actual records in each full component, rather than assuming all files include all selected plans. Formulary records without a price remain visible for investigation.

In [4]:
table(json.loads((E/'plan_coverage.json').read_text()))
plans=query('SELECT contract_id,plan_id,segment_id,plan_name,premium,deductible,formulary_id,pdp_region_code FROM clean.plans ORDER BY contract_id,plan_id')
assert len(plans)==12
assert all(p['pdp_region_code']=='13' and p['contract_id'].startswith('S') for p in plans)
table(plans)

contract_id,plan_id,plan_name,formulary_rows,pricing_rows,network_rows,cost_sharing_rows,formulary_rows_without_price
S4802,084,Wellcare Classic (PDP),2992,8976,54037,28,0
S4802,148,Wellcare Value Script (PDP),3266,9798,54037,40,0
S5584,001,Prescription Blue Select (PDP),3085,6170,56390,24,0
S5584,002,Prescription Blue Premium (PDP),3085,6170,56390,24,0
S5601,026,SilverScript Choice (PDP),3405,10215,60521,28,0
S5617,221,HealthSpring Assurance Rx (PDP),3287,9861,53998,28,0
S5617,363,HealthSpring Extra Rx (PDP),3370,10110,53998,34,0
S5884,136,Humana Basic Rx Plan (PDP),3052,6104,55048,23,0
S5884,159,Humana Premier Rx Plan (PDP),3156,6312,55048,24,0
S5884,192,Humana Value Rx Plan (PDP),3089,6178,55048,27,0


contract_id,plan_id,segment_id,plan_name,premium,deductible,formulary_id,pdp_region_code
S4802,084,000,Wellcare Classic (PDP),0.000000,615.000000,00026191,13
S4802,148,000,Wellcare Value Script (PDP),0.000000,615.000000,00026195,13
S5584,001,000,Prescription Blue Select (PDP),78.400000,615.000000,00026449,13
S5584,002,000,Prescription Blue Premium (PDP),106.700000,0.000000,00026449,13
S5601,026,000,SilverScript Choice (PDP),78.300000,615.000000,00026007,13
S5617,221,000,HealthSpring Assurance Rx (PDP),0.000000,615.000000,00026089,13
S5617,363,000,HealthSpring Extra Rx (PDP),50.400000,615.000000,00026094,13
S5884,136,000,Humana Basic Rx Plan (PDP),0.000000,615.000000,00026399,13
S5884,159,000,Humana Premier Rx Plan (PDP),98.100000,0.000000,00026402,13
S5884,192,000,Humana Value Rx Plan (PDP),0.000000,601.000000,00026400,13


## 4. Join examples

Plan → formulary uses formulary_id within the frozen year/quarter. Pricing additionally requires contract/plan/segment, NDC and days supply. Cost-sharing requires the same plan, tier, coverage phase and translated days supply. Examples below use initial coverage and 30 days to verify linkage, not to calculate annual costs. There may be multiple NDCs per clinical drug; those alternatives remain explicit. The full join has 22,850 priced combinations without initial cost-sharing records, all at 60/90 days. These are marked missing_cost_sharing, not estimated by borrowing a 30-day rate. All 30-day priced combinations have a matching initial cost-sharing row.

In [5]:
examples=json.loads((E/'join_examples.json').read_text())
# One example per plan/product for readability; complete joined examples remain saved in evidence.
shown=[];seen=set()
for row in examples:
    key=(row['contract_id'],row['plan_id'],row['rxcui'])
    if key not in seen:
        seen.add(key);shown.append(row)
table(shown)
table([json.loads((E/'integration_summary.json').read_text())['join_examples']])
table(query("SELECT days_supply,input_status,count(*) AS records FROM clean.plan_drug_cost_inputs GROUP BY ALL ORDER BY days_supply,input_status"))

contract_id,plan_id,segment_id,plan_name,drug_name,rxcui,ndc,tier,days_supply,unit_cost,premium,deductible,prior_authorization_yn,step_therapy_yn,quantity_limit_yn,coverage_level,cost_type_pref,cost_amt_pref,cost_type_nonpref,cost_amt_nonpref,ded_applies_yn
S4802,084,000,Wellcare Classic (PDP),lisinopril 10 MG Oral Tablet,314076,68180098003,1,30,0.073000,0.000000,615.000000,N,N,N,1,1,0.000000,1,10.000000,Y
S4802,084,000,Wellcare Classic (PDP),atorvastatin 20 MG Oral Tablet,617310,60505257908,1,30,0.147600,0.000000,615.000000,N,N,Y,1,1,0.000000,1,10.000000,Y
S4802,084,000,Wellcare Classic (PDP),metformin hydrochloride 500 MG Oral Tablet,861007,70010006310,1,30,0.045200,0.000000,615.000000,N,N,Y,1,1,0.000000,1,10.000000,Y
S4802,148,000,Wellcare Value Script (PDP),lisinopril 10 MG Oral Tablet,314076,68180098003,1,30,0.035400,0.000000,615.000000,N,N,N,1,1,0.000000,1,15.000000,N
S4802,148,000,Wellcare Value Script (PDP),atorvastatin 20 MG Oral Tablet,617310,60505257908,1,30,0.109600,0.000000,615.000000,N,N,Y,1,1,0.000000,1,15.000000,N
S4802,148,000,Wellcare Value Script (PDP),metformin hydrochloride 500 MG Oral Tablet,861007,70010006310,1,30,0.026200,0.000000,615.000000,N,N,Y,1,1,0.000000,1,15.000000,N
S5584,001,000,Prescription Blue Select (PDP),lisinopril 10 MG Oral Tablet,314076,68180098003,1,30,0.043400,78.400000,615.000000,N,N,Y,1,1,1.000000,1,6.000000,Y
S5584,001,000,Prescription Blue Select (PDP),atorvastatin 20 MG Oral Tablet,617310,60505257908,1,30,0.087600,78.400000,615.000000,N,N,Y,1,1,1.000000,1,6.000000,Y
S5584,001,000,Prescription Blue Select (PDP),metformin hydrochloride 500 MG Oral Tablet,861007,70010006310,1,30,0.040000,78.400000,615.000000,N,N,Y,1,1,1.000000,1,6.000000,Y
S5584,002,000,Prescription Blue Premium (PDP),lisinopril 10 MG Oral Tablet,314076,68180098003,1,30,0.043400,106.700000,0.000000,N,N,Y,1,1,1.000000,1,6.000000,N


records,distinct_plans,distinct_drugs,missing_initial_cost_share
36,12,3,0


days_supply,input_status,records
30,joined_not_annual_estimate,38526
60,joined_not_annual_estimate,12569
60,missing_cost_sharing,10490
90,joined_not_annual_estimate,26166
90,missing_cost_sharing,12360


## 5. Readable drug names and conservative matching

CMS codes are linked to NLM RxNorm product names by exact RXCUI. Full product names include strength/form and brand when present. The current NLM catalog version is saved separately from the historical CMS snapshot. Matching is identity lookup, not substitution advice.

Incomplete names return candidates. Missing entries are flagged. A unique RXCUI with several NDCs still needs a defined product selection; the system does not automatically use the cheapest code.

In [6]:
print('NLM catalog version:',json.loads((ROOT/'data/reference/rxnorm_version.json').read_text()))
table(query('SELECT name_status,count(*) AS drug_concepts FROM clean.drug_catalog GROUP BY name_status'))
table(query("SELECT rxcui,name,tty FROM clean.drug_catalog WHERE contains(lower(name),'atorvastatin') ORDER BY name LIMIT 10"))
for value,kind in [('atorvastatin','name'),('atorvastatin 20 MG Oral Tablet','name'),('unknown-example-drug','name'),('123','ndc')]:
    result=match_medication(value,kind)
    print(value, '->', result['status'])
    if result.get('candidates'):table(result['candidates'][:5])

NLM catalog version: {'version': '02-Nov-2026', 'apiVersion': '3.1.357'}


name_status,drug_concepts
mapped,4261


rxcui,name,tty
597987,amlodipine 10 MG / atorvastatin 10 MG Oral Tablet,SCD
597967,amlodipine 10 MG / atorvastatin 20 MG Oral Tablet,SCD
597990,amlodipine 10 MG / atorvastatin 40 MG Oral Tablet,SCD
404013,amlodipine 10 MG / atorvastatin 80 MG Oral Tablet,SCD
597971,amlodipine 2.5 MG / atorvastatin 10 MG Oral Tablet,SCD
597974,amlodipine 2.5 MG / atorvastatin 20 MG Oral Tablet,SCD
597993,amlodipine 2.5 MG / atorvastatin 40 MG Oral Tablet,SCD
597977,amlodipine 5 MG / atorvastatin 10 MG Oral Tablet,SCD
597980,amlodipine 5 MG / atorvastatin 20 MG Oral Tablet,SCD
597984,amlodipine 5 MG / atorvastatin 40 MG Oral Tablet,SCD


atorvastatin -> needs_clarification


rxcui,name,tty
597987,amlodipine 10 MG / atorvastatin 10 MG Oral Tablet,SCD
597967,amlodipine 10 MG / atorvastatin 20 MG Oral Tablet,SCD
597990,amlodipine 10 MG / atorvastatin 40 MG Oral Tablet,SCD
404013,amlodipine 10 MG / atorvastatin 80 MG Oral Tablet,SCD
597971,amlodipine 2.5 MG / atorvastatin 10 MG Oral Tablet,SCD


atorvastatin 20 MG Oral Tablet -> matched


rxcui,name,tty
617310,atorvastatin 20 MG Oral Tablet,SCD


unknown-example-drug -> unmatched
123 -> needs_clarification


## 6. Pharmacy network and directory demonstration

The complete CMS network subset supplies membership and preferred flags. A bounded NPPES lookup supplies current names/addresses for up to 20 specific Michigan locations. This is an intake demonstration, not a complete directory or a claim that current directory details equal Q3 historical details. Network status comes from CMS, not the directory.

In [7]:
table(query('SELECT pharmacy_number,npi,name,address,city,state,source_network_zip FROM clean.pharmacy_directory ORDER BY name'))
table(query("SELECT p.plan_name,n.pharmacy_number,d.name,n.pharmacy_zipcode,n.pharmacy_retail,n.preferred_status_retail,n.pharmacy_mail,n.preferred_status_mail FROM clean.pharmacy_network n JOIN clean.plans p USING(contract_id,plan_id,segment_id) JOIN clean.pharmacy_directory d USING(pharmacy_number) ORDER BY n.pharmacy_number,p.plan_name LIMIT 24"))

pharmacy_number,npi,name,address,city,state,source_network_zip
101922895010,1922895010,ALGONAC PHARMACY,423 MICHIGAN ST,ALGONAC,MI,48001
101154841526,1154841526,ALLIANCE SPECIALTY PHARMACY LLC,25301 VAN DYKE AVE,CENTER LINE,MI,48015
101316639958,1316639958,ALMONT PHARMACY LLC,4545 VAN DYKE RD,ALMONT,MI,48003
101538424858,1538424858,CENTERLINE CITY PHARMACY LLC,8033 E 10 MILE RD STE 103,CENTER LINE,MI,48015
101386065126,1386065126,CLAWSON CARE PHARMACY LLC,117 W 14 MILE RD,CLAWSON,MI,48017
101144567843,1144567843,EASTPOINTE MEDICAL PHARMACY LLC,22640 KELLY RD,EASTPOINTE,MI,48021
101316568645,1316568645,EHARDT'S PHARMACY INCORPORATED,3433 CAPAC RD STE 3,MUSSEY,MI,48014
101245652742,1245652742,ELWASEEM -MOORE INC,18065 E 8 MILE RD,EASTPOINTE,MI,48021
101649470568,1649470568,KROGER CO OF MICHIGAN,2600 POINTE TREMBLE RD,ALGONAC,MI,48001
101679507420,1679507420,KROGER CO OF MICHIGAN,685 E MAPLE RD,BIRMINGHAM,MI,48009


plan_name,pharmacy_number,name,pharmacy_zipcode,pharmacy_retail,preferred_status_retail,pharmacy_mail,preferred_status_mail
AARP Medicare Rx Preferred from UHC (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,N,N,N
AARP Medicare Rx Saver from UHC (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,N,N,N
HealthSpring Assurance Rx (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,Y,N,N
HealthSpring Extra Rx (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,Y,N,N
Humana Basic Rx Plan (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,N,N,N
Humana Premier Rx Plan (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,N,N,N
Humana Value Rx Plan (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,N,N,N
SilverScript Choice (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,N,N,N
Wellcare Classic (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,N,N,N
Wellcare Value Script (PDP),101144567843,EASTPOINTE MEDICAL PHARMACY LLC,48021,Y,N,N,N


## 7. Intake requirements and remaining boundaries

- Required calculation inputs: exact medication product, quantity with unit, days supplied and fill schedule; plan and pharmacy selection.
- Michigan geography is fixed for the prototype. A beneficiary ZIP lookup is separate from pharmacy ZIP.
- Retail pharmacy flexibility and explanation preferences tailor the brief; they do not change medication identity.
- Preserve brand/generic, strength, formulation and route distinctions. Unknown/ambiguous selections require clarification.
- Average retail unit cost is not an exact pharmacy-specific quote or a validated mail-order price.
- Extra Help, income-based adjustments, annual benefit-phase simulation, special-drug rules and monthly budget feasibility need separate rules and validation. Data integration alone does not complete those calculations.
- Taylor's intake v2 is now connected in PlanWise_Intake_CMS_Connected.ipynb. It preserves the three tiers and produces 24 plan-linked records for two real drug products and a verified pharmacy. Twelve validation checks passed. Annual cost calculations remain future work.
- Two standup comments and review of team feedback remain pending the relevant posts/channel context. No comments have been posted from this artifact.

See `docs/INTAKE_AND_MATCHING.md` for the complete field-to-question mapping and `docs/DATA_DICTIONARY.md` for types and join rules.